# Task 2 - detection

Fine-tune the detector on the 3-class set from Task 1 and score every run the same way.

| Run | Model | Size | Purpose |
| --- | --- | --- | --- |
| B0 | COCO weights, no fine-tune | 640 | the floor. It can only find stop signs, so cone and barrier AP are 0 by construction |
| B1 | `yolo26n` fine-tuned | 640 | main deployable candidate |
| B1b | `yolo11n` fine-tuned | 640 | other family, for the latency pick in Task 4 |
| B2 | `yolo26s` fine-tuned | 960 | accuracy ceiling and pseudo-label teacher |

Optional accuracy levers: `B1bal` (class-balanced sampling) and `B1u` (unfreeze the backbone at a lower LR).

**Why a custom evaluator.** `model.val()` cannot score B0: that model predicts 80 COCO classes
against a 3-class set, so the indices do not line up. `eval/det_eval.py` implements COCO-style AP
once and runs every variant through it, remapping the zero-shot predictions, so the comparison is
apples to apples. `--cross-check` also prints `model.val()` for the fine-tuned runs.

## Setup

Runs locally. `python -m common.doctor` says whether torch sees a GPU and whether the
dataset is built; the next cell builds it if it is not.

Rough timings with a mid-range GPU: B1 about 45 min, B1b about the same, B2 (s at 960)
2-3 h. Drop `epochs` with `--set epochs=40` if that is too long. On CPU, train B1 only
and expect hours - the Task 4 benchmarks are where CPU numbers actually matter.


In [ ]:
# Setup. A local checkout is the normal case; the Kaggle branch is a fallback.
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/heller007/eric-robotics-assessment.git"

if Path("/kaggle/working").exists():
    REPO = Path("/kaggle/working/eric-robotics-assessment")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    !pip install -q ultralytics kagglehub
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO)
sys.path.insert(0, str(REPO))
print("repo:", REPO)

In [ ]:
# What this machine can do, and whether the dataset is here yet.
!python -m common.doctor

In [ ]:
# The dataset Task 1 builds. Already-built datasets are only re-validated,
# so re-running this is cheap. Point --nuscenes-root at a local copy of
# v1.0-mini, or use --fetch-nuscenes to download it.
!python -m data.prepare --draw 6

In [ ]:
# The floor: COCO weights, no fine-tuning. Everything later has to beat this.
!python -m eval.det_eval --run B0 --device 0

In [ ]:
# B1 - the main candidate. freeze=10 for the first pass: ~2,400 images overfit fast.
!python -m detection.train --run B1 --device 0

In [ ]:
!python -m eval.det_eval --weights runs/B1/weights/best.pt --name B1 --device 0 --cross-check

In [ ]:
# B1b - same recipe, other family. Task 4 picks between them on measured latency.
!python -m detection.train --run B1b --device 0
!python -m eval.det_eval --weights runs/B1b/weights/best.pt --name B1b --device 0

In [ ]:
# B2 - accuracy ceiling at 960 px. The long one; shorten with --set epochs=60 if needed.
!python -m detection.train --run B2 --device 0
!python -m eval.det_eval --weights runs/B2/weights/best.pt --name B2 --imgsz 960 --device 0

### Accuracy levers (optional)

Run these only if B1 leaves a gap worth closing. Cheapest first.

In [ ]:
# Lever 4: repeat images so each class is seen a similar number of times per epoch.
!python -m data.balance --threshold 0.5
!python -m detection.train --run B1bal --device 0
!python -m eval.det_eval --weights runs/B1bal/weights/best.pt --name B1bal --device 0

In [ ]:
# Lever 5: unfreeze the backbone and continue from B1 at a lower learning rate.
!python -m detection.train --run B1u --device 0
!python -m eval.det_eval --weights runs/B1u/weights/best.pt --name B1u --device 0

In [ ]:
# Lever 2: input size on fixed weights. Far cones are 10-20 px tall at 640.
for size in (640, 800, 960):
    !python -m eval.det_eval --weights runs/B1/weights/best.pt --name B1_imgsz{size} --imgsz {size} --device 0

In [ ]:
# Every variant scored so far, newest last.
import pandas as pd

df = pd.read_csv("results/detection.csv")
display(df)
print("\nbest mAP50:", df.loc[df.mAP50.idxmax(), "variant"])

In [ ]:
# Per-class detail for B1, including recall by ground-truth range - this is where
# far cones either show up or do not.
import json

detail = json.loads(Path("results/det_eval_B1.json").read_text())
for name, m in detail["per_class"].items():
    print(f"{name:<10} AP50 {m['AP50']:.3f}  AP50-95 {m['AP50_95']:.3f}  gt {m['gt_instances']}")
    for rng, v in m["recall_by_range"].items():
        if v["recall"] is not None:
            print(f"             {rng:<10} recall {v['recall']:.2f}  (n={v['gt']})")

In [ ]:
# Qualitative check: what the detector actually draws on held-out frames.
from IPython.display import Image, display
from common.env import writable_root

VAL_IMAGES = writable_root() / "datasets/yolo_3class/images/val"
!python -m detection.infer --weights runs/B1/weights/best.pt --source "{VAL_IMAGES}" --limit 6 --dest results/predictions/B1 --device 0

for p in sorted(Path("results/predictions/B1").glob("*.jpg"))[:6]:
    display(Image(str(p), width=900))

## Done

Before moving on, confirm:

- every fine-tuned run beats B0 on **all three** classes;
- `--cross-check` agrees with the custom evaluator to within a point or two (small differences are
  expected: ultralytics letterboxes and rectangles differently);
- stop sign AP is not near zero - if it is, the pseudo-labels from Task 1 are the first suspect;
- the per-range recall for cones past 20 m is understood, even if it is low. It feeds the decision
  about input size and the P2 head.

`runs/B1/weights/best.pt` is what Task 3 and Task 4 start from. Weights are git-ignored, so if the
next task runs on the other machine, copy that file across or publish it as a dataset - only the
code travels through git.

Next: `task3_distance.ipynb` - fit the priors, then D0 / D1 / fusion on oracle boxes and on these
predictions.